https://www.analyticsvidhya.com/blog/2024/01/xgboost-for-time-series-forecasting/#:~:text=Preparing%20Data%20for%20Time-Series%20Forecasting%20with%20XGBoost%201,3%20Step%203%3A%20Handling%20Missing%20Values%20and%20Outliers

What is XGBoost?
XGBoost, short for Extreme Gradient Boosting, is a powerful machine learning algorithm that excels in various predictive modeling tasks, including time-series forecasting. It is an ensemble learning method that combines the predictions of multiple weak models (decision trees) to create a strong predictive model. XGBoost is known for its scalability, speed, and ability to handle complex relationships in the data.

Advantages of XGBoost for Time-Series Forecasting
XGBoost offers several advantages that make it an excellent choice for time-series forecasting:

Handling Non-Linear Relationships: XGBoost can capture complex non-linear relationships between input features and the target variable, making it suitable for time-series data with intricate patterns.
Feature Importance: XGBoost provides insights into the importance of different features, allowing analysts to identify the most influential factors in the time-series data.
Regularization: XGBoost incorporates regularization techniques to prevent overfitting, ensuring that the model generalizes well to unseen data.
Handling Missing Values and Outliers: XGBoost can handle missing values and outliers in the data, reducing the need for extensive data preprocessing.

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import catboost

# http://lotto.merseyworld.com/cgi-bin/lottery?days=20&Machine=Z&Ballset=0&order=0&show=1&year=0&display=CSV

ModuleNotFoundError: No module named 'catboost'

In [11]:
df = pd.read_csv("Euromillones.csv")

In [12]:
# Define a correct parse function for your date format
def parse_date(x):
    return pd.to_datetime(x, format="%d,%b,%Y")


# Apply the parse function to combine the columns into a datetime column
df["YYYY_MMM_DD"] = df.apply(
    lambda row: parse_date(f"{row['DD']},{row['MMM']},{row['YYYY']}"), axis=1
)

# Set the new 'Date' column as index if needed
df.set_index("YYYY_MMM_DD", inplace=True)
df.drop(columns=["DD", "MMM", "YYYY"], inplace=True)
# Eliminar espacios en blanco del nombre de las columnas
df = df.rename(columns=lambda x: x.strip())
df = df.iloc[::-1]  # reverse order of rows

In [13]:
df = df.drop(["No.", "Day", "Jackpot", "Wins"], axis=1)
df.columns = ["N1", "N2", "N3", "N4", "N5", "S1", "S2"]

In [14]:
df[["N1", "N2", "N3", "N4", "N5"]]

,N1,N2,N3,N4,N5
YYYY_MMM_DD,,,,,
2004-02-13,32,16,29,41,36
2004-02-20,13,50,47,7,39
2004-02-27,37,19,18,14,31
2004-03-05,39,37,4,7,33
2004-03-12,44,47,15,28,24
...,...,...,...,...,...
2024-05-21,13,11,48,14,34
2024-05-24,12,9,22,18,50
2024-05-28,18,16,35,41,36


Preparing Data for Time-Series Forecasting with XGBoost
Step 1: Data Cleaning and Preprocessing
Before applying XGBoost to time-series data, it is essential to clean and preprocess the data. This involves handling missing values, removing outliers, and ensuring the data is in the correct format. For example, if the time-series data has irregular time intervals, it requires resamplin to ensure a consistent time interval.

Also Read: Data Cleaning for Beginners- Why and How ?

Step 2: Feature Engineering for Time-Series Data
Feature engineering plays a crucial role in time-series forecasting with XGBoost. It involves creating relevant features from the raw data that capture the underlying patterns and trends. Some common techniques include lag features (using past values as predictors), rolling statistics (e.g., moving averages), and Fourier transformations to capture seasonality.



In [17]:
stacked_df = (df.stack().reset_index(name="Valor").drop("level_1", axis=1))
stacked_df

,YYYY_MMM_DD,Valor
0,2004-02-13,32
1,2004-02-13,16
2,2004-02-13,29
3,2004-02-13,41
4,2004-02-13,36
...,...,...
12189,2024-06-04,6
12190,2024-06-04,9
12191,2024-06-04,14
12192,2024-06-04,4


Lag Features
Lag features involve incorporating past values of the target variable as predictors. The create_lag_features function in the provided code generates lag features up to a specified number of time steps (lag_steps). This technique allows the model to capture temporal dependencies and historical trends within the time-series data.



In [22]:
# Creating lag features for time-series data

def create_lag_features(data, lag_steps=7):

    for i in range(1, lag_steps + 1):

        data[f'lag_{i}'] = data['Valor'].shift(i)

    return data

In [23]:

# Applying lag feature creation to the dataset

lagged_data = create_lag_features(stacked_df, lag_steps=3)

Rolling Mean
The rolling mean is a technique that smoothens time-series data by calculating the average over a specified window of observations. The create_rolling_mean function creates a new feature, ‘rolling_mean,’ by computing the mean of the target variable over a user-defined window size. This helps to highlight trends and patterns by reducing noise and fluctuations in the data.

In [24]:
# Creating rolling mean for time-series data

def create_rolling_mean(data, window_size=3):

    data['rolling_mean'] = data['Valor'].rolling(window=window_size).mean()

    return data

In [26]:
# Applying rolling mean to the dataset

rolled_data = create_rolling_mean(stacked_df, window_size=5)

Fourier Transformation
Fourier transformation is applied to capture periodic components or seasonality within time-series data. The apply_fourier_transform function uses the Fast Fourier Transform (FFT) to convert the target variable values into the frequency domain. The resulting ‘fourier_transform’ feature contains information about the amplitudes of different frequency components, aiding in the identification and modeling of cyclic patterns in the time series.

In [29]:
# Applying Fourier transformation for capturing seasonality
from scipy.fft import fft

def apply_fourier_transform(data):

    values = data['Valor'].values

    fourier_transform = fft(values)

    data['fourier_transform'] = np.abs(fourier_transform)

    return data

In [30]:
# Applying Fourier transformation to the dataset

fourier_data = apply_fourier_transform(stacked_df)

Step 3: Handling Missing Values and Outliers
XGBoost can handle missing values and outliers in the data. Missing values can be imputed using techniques such as interpolation or mean imputation. Outliers can be detected and treated using robust statistical methods or by transforming the data. By handling missing values and outliers effectively, XGBoost can provide more accurate forecasts.



Building and Training an XGBoost Model for Time-Series Forecasting


Step 1: Splitting the Data into Training and Testing Sets
To assess the performance of the XGBoost model, one must partition the time-series data into training and testing sets. The training set facilitates model training, and the testing set enables the evaluation of its performance on unseen data. Preserving the temporal order of observations is crucial when splitting the data

In [ ]:
# Splitting time-series data into training and testing sets

train_size = int(len(data) * 0.8)

train_data, test_data = data[:train_size], data[train_size:]